# Day 1: LLM and embedding fundamentals

Notebook for Day 1 of the course (`course/week-1/day-01-llm-fundamentals/`).

## Step 1: Tokens

**Question:** what does an LLM actually read?

LLMs read **tokens** (pieces of words), not words. Context windows, chunk sizes, cost and speed are all counted in tokens.

`tiktoken` uses OpenAI's tokenizers. Llama's tokenizer is a little different, so these counts are close estimates. That's enough for sizing chunks.

In [8]:
import tiktoken

# o200k_base: the tokenizer used by gpt-4o. Use the same one everywhere in this project.
encoding = tiktoken.encoding_for_model("gpt-4o")
print(encoding.name)

o200k_base


### 1a. How are words split?

`repr()` shows hidden spaces at the start of tokens.

In [23]:
def show_tokens(text: str) -> None:
    """Print the word count, token count and each token piece."""
    token_ids = encoding.encode(text)
    print(f"Text:   {text!r}")
    print(f"Words:  {len(text.split())}")
    print(f"Tokens: {len(token_ids)}")
    for token_id in token_ids:
        print(f"  {token_id:>7}  {encoding.decode([token_id])!r}")
    print()


show_tokens("LoRA")
show_tokens("LoRA freezes the pretrained weights and trains small adapter matrices. Unbelievable!")
show_tokens("We use LoRA to fine-tune the model.")

Text:   'LoRA'
Words:  1
Tokens: 2
     6681  'Lo'
     7896  'RA'

Text:   'LoRA freezes the pretrained weights and trains small adapter matrices. Unbelievable!'
Words:  11
Tokens: 16
     6681  'Lo'
     7896  'RA'
   178077  ' freezes'
      290  ' the'
   191451  ' pretrained'
    28460  ' weights'
      326  ' and'
    46841  ' trains'
     3291  ' small'
    20725  ' adapter'
    75935  ' matrices'
       13  '.'
     1367  ' Un'
     9880  'bel'
    45794  'ievable'
        0  '!'

Text:   'We use LoRA to fine-tune the model.'
Words:  7
Tokens: 11
     2167  'We'
     1199  ' use'
     8786  ' Lo'
     7896  'RA'
      316  ' to'
     8975  ' fine'
     2378  '-t'
     2561  'une'
      290  ' the'
     2359  ' model'
       13  '.'



### 1b. Words-to-tokens ratio on a real paper

Rule of thumb for English: about **0.75 words per token** (1 token ≈ ¾ of a word).

In [10]:
arxiv_paragraph = "Unequal couplings to a common emergent gauge field provide a continuous way to tune competing magnetic instabilities in an effective description of U(1) spin-liquid mixtures. We study this mechanism in quantum electrodynamics in 2+1 dimensions with two families of four-component Dirac fermions. Combining a leading large-N treatment with Schwinger--Dyson equations, we obtain chiral condensates representing Néel order in the two sectors. At fixed fermion number, the coupling ratio κ=e22/e21 controls which condensates develop. Depending on N, the phases with a single condensate are connected through either coexistence or a finite interval in which both fermion families remain gapless. The mass-generation thresholds are reciprocal, and the fermion gaps close with Miransky essential singularities. The renormalization group flow has a circular line of fixed points parametrized by the invariant ratio κ. At the calculated order, gauge-invariant staggered magnetization correlations have continuously varying exponents in the gapless regime, while the exponent for correlations between the two sectors is independent of κ. These results connect the competition between magnetic orders to a continuous family of critical theories and identify magnetic correlation signatures of the intervening gapless regime."

In [11]:
paragraph_tokens = encoding.encode(arxiv_paragraph)
word_count = len(arxiv_paragraph.split())
token_count = len(paragraph_tokens)

print(f"Words:           {word_count}")
print(f"Tokens:          {token_count}")
print(f"Words per token: {word_count / token_count:.2f}")
print(f"Tokens per word: {token_count / word_count:.2f}")

Words:           183
Tokens:          249
Words per token: 0.73
Tokens per word: 1.36


In [12]:
# Which words cost the most tokens? Jargon and symbols usually top the list.
word_costs = []
for word in set(arxiv_paragraph.split()):
    n_tokens = len(encoding.encode(" " + word))  # leading space: words mid-sentence carry one
    word_costs.append((n_tokens, word))

word_costs.sort(reverse=True)  # tuples sort by their first item, the token count

for n_tokens, word in word_costs[:10]:
    print(f"{n_tokens:>2} tokens  {word}")

 6 tokens  Schwinger--Dyson
 5 tokens  κ=e22/e21
 4 tokens  U(1)
 4 tokens  2+1
 3 tokens  singularities.
 3 tokens  renormalization
 3 tokens  gauge-invariant
 3 tokens  gapless.
 3 tokens  fermions.
 3 tokens  electrodynamics


### Findings

- **"LoRA" is 2 tokens** (`Lo` + `RA`). Rare words and unusual capitalization get split into familiar pieces.
- **Spaces belong to the next word:** `' the'` is one token. The same word can become different tokens at the start of a sentence and in the middle.
- **Paper paragraph: 183 words, 249 tokens = 0.73 words per token**, right on the rule of thumb. Jargon and symbols (`κ=e22/e21`, `Schwinger--Dyson`) cost several tokens each, but the plain words around them balance it out.
- **Why it matters:** chunk sizes and context windows are set in tokens. A 200-token chunk is about 150 words.

## Step 2: Calling the LLM

In [24]:
import json
import httpx

url = "http://localhost:11434/api/generate"

body = {
    "model": "llama3.1:8b",                   # use "llama3.2:3b" if your Mac has 8 GB of memory
    "prompt": "What is LoRA in one sentence?",
    "stream": False,                          # get the whole answer at once
}

response = httpx.post(url, json=body, timeout=120)
response.raise_for_status()                   # stop with a clear error if Ollama returned one

data = response.json()                        # the reply as a Python dictionary

print(json.dumps(data, indent=2))
# print(data)

{
  "model": "llama3.1:8b",
  "created_at": "2026-11-18T23:09:37.819766Z",
  "response": "LoRA (Learning to Reconcile Activations) is a lightweight and efficient neural network adaptation technique that enables fast model updates and fine-tuning by learning a set of small, adjustable parameters that can be applied to a pre-trained model.",
  "done": true,
  "done_reason": "stop",
  "context": [
    128006,
    882,
    128007,
    271,
    3923,
    374,
    6621,
    5726,
    304,
    832,
    11914,
    30,
    128009,
    128006,
    78191,
    128007,
    271,
    4357,
    5726,
    320,
    48567,
    311,
    1050,
    41546,
    458,
    15050,
    811,
    8,
    374,
    264,
    30244,
    323,
    11297,
    30828,
    4009,
    34185,
    15105,
    430,
    20682,
    5043,
    1646,
    9013,
    323,
    7060,
    2442,
    38302,
    555,
    6975,
    264,
    743,
    315,
    2678,
    11,
    37030,
    5137,
    430,
    649,
    387,
    9435,
    311,
    264,


In [ ]:
print(data["prompt_eval_count"])
print(data["eval_count"])

18
54


In [ ]:
def ask(prompt: str, temperature: float = 0) -> str:
    """Send a prompt to Ollama, print speed stats, and return the answer text."""
    url = "http://localhost:11434/api/generate"
    body = {
        "model": "llama3.1:8b",
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature},
    }

    response = httpx.post(url, json=body, timeout=120)
    response.raise_for_status()
    data = response.json()

    seconds = data["eval_duration"] / 1_000_000_000   # nanoseconds -> seconds
    tokens = data["eval_count"]
    tokens_per_second = tokens / seconds

    print(f"seconds taken:     {seconds:.2f}")
    print(f"tokens generated:  {tokens}")
    print(f"tokens per second: {tokens_per_second:.1f}")

    return data["response"]


answer = ask("Who is Steve Jobs? Answer in two sentences.")
print()
print(answer)

In [42]:
print(ask("What does the acronym LoRA stand for in machine learning?"))

seconds taken:     3.97
tokens generated:  183
tokens per second: 46.1
In machine learning, LoRA stands for "Learning to be Optimal" or more specifically, "Locally Optimized Weights for Rapid Adaptation".

However, the more widely accepted and used meaning of LoRA is "Locally Optimized Weights for Rapid Adaptation" or "Locally Optimized Weights for Adaptation", but it is more commonly referred to as "Locally Optimized Weights" or simply "LoRA".

LoRA is a technique used in machine learning, particularly in deep learning, to adapt pre-trained models to new tasks or domains with minimal additional training data. It involves adding a set of learnable weights to the pre-trained model, which can be fine-tuned to adapt to the new task or domain.

LoRA was introduced in the paper "LoRA: Low-Rank Adaptation for Neural Architectures" by You et al. in 2020.


### Findings
- Ollama returns JSON: the answer is in `response`, token counts in `eval_count`, durations in nanoseconds.
- My Mac generates about 46 tokens/second with llama3.1:8b. The first call is slower because the model loads into memory (`load_duration`).
- Hallucination example: asked what LoRA stands for, the model invented "Locally Optimized Weights for Rapid Adaptation"
  and cited "You et al., 2020". Correct: Low-Rank Adaptation, Hu et al., 2021.

## Step 3: Temperature

In [54]:
prompt = "Give me a one-line name idea for a coffee shop run by robots."

for temperature in [0, 1]:          # outer loop: the temperature we test
    for run in range(1, 4):         # inner loop: 3 runs at that same temperature
        answer = ask(prompt, temperature)
        print(f"T={temperature}, run {run}: {answer.strip()}")
        print()
    print("-" * 60)                 # separator between the T=0 group and the T=1 group

seconds taken:     0.37
tokens generated:  15
tokens per second: 41.1
T=0, run 1: Here's a one-line name idea: "Brewtronics"

seconds taken:     0.30
tokens generated:  15
tokens per second: 49.4
T=0, run 2: Here's a one-line name idea: "Brewtronics"

seconds taken:     0.30
tokens generated:  15
tokens per second: 49.2
T=0, run 3: Here's a one-line name idea: "Brewtronics"

------------------------------------------------------------
seconds taken:     0.11
tokens generated:  6
tokens per second: 56.4
T=1, run 1: "Gear Grounds"

seconds taken:     0.11
tokens generated:  6
tokens per second: 55.4
T=1, run 2: "Brewtronics"

seconds taken:     0.50
tokens generated:  24
tokens per second: 48.2
T=1, run 3: Here's a one-line name idea for a coffee shop run by robots: "Brew-tal Automation Co."

------------------------------------------------------------


### Findings
**Temperature controls randomness, not accuracy.** At each step the model has a list of possible next tokens with probabilities; temperature decides how it picks one.
- **T=0:** always the most likely token, so (nearly) the same answer every time. Use for RAG answers and evals, which must be repeatable.
- **T between 0 and 1:** still sensible, with some variety. T=1 uses the model's natural probabilities; 0.7–1 suits creative work.
- **T above 1 (e.g. 2):** probabilities get flattened, so unlikely words come up often and the text can stop making sense.
- A low temperature doesn't make answers truer: at T=0 the model would still get LoRA wrong, just consistently. Accuracy comes from giving it the right context (RAG).

**What I saw when I ran it** (fill in after running the cell above):
- T=0, 3 runs: _identical? yes / no_
- T=1, 3 runs: _how different?_


## Step 4: Embeddings

In [59]:
from sentence_transformers import SentenceTransformer, util

# 1. Load the embedding model (downloads about 90 MB the first time)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2. Six sentences: two per topic, with each pair worded differently
sentences = [
    "The cat sat on the mat.",
    "A kitten was resting on the rug.",
    "The stock market fell 3% today.",
    "Shares dropped sharply this afternoon.",
    "LoRA fine-tunes large models by training small adapter matrices.",
    "Low-rank adaptation reduces the cost of fine-tuning language models.",
]
labels = ["cat 1", "cat 2", "finance 1", "finance 2", "lora 1", "lora 2"]

# 3. Turn each sentence into a vector of numbers
embeddings = model.encode(sentences, normalize_embeddings=True)
print("Shape:", embeddings.shape)   # (number of sentences, numbers per sentence)
print("First 5 numbers of 'cat 1':", embeddings[0][:5])
print()

# 4. Compare every sentence with every other sentence
similarity = util.cos_sim(embeddings, embeddings)

# 5. Print the grid as a readable table
print(" " * 11, end="")
for label in labels:
    print(f"{label:>10}", end="")
print()

for row in range(len(labels)):
    print(f"{labels[row]:>10} ", end="")
    for col in range(len(labels)):
        print(f"{similarity[row][col].item():>10.2f}", end="")
    print()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9511.73it/s]


Shape: (6, 384)
First 5 numbers of 'cat 1': [ 0.1302372  -0.01577287 -0.03671673  0.05798635 -0.05979172]

                cat 1     cat 2 finance 1 finance 2    lora 1    lora 2
     cat 1       1.00      0.63      0.09      0.03      0.01      0.03
     cat 2       0.63      1.00      0.10      0.04      0.03      0.03
 finance 1       0.09      0.10      1.00      0.59     -0.01      0.03
 finance 2       0.03      0.04      0.59      1.00      0.02      0.06
    lora 1       0.01      0.03     -0.01      0.02      1.00      0.40
    lora 2       0.03      0.03      0.03      0.06      0.40      1.00


### Findings
- Each sentence becomes 384 numbers (the embedding dimension). Shape (6, 384).
- Paraphrases with different words still match: cats 0.63, finance 0.59. Unrelated topics score about 0–0.1.
- Real paraphrase scores are about 0.5–0.8, not near 1.0. Thresholds must come from data.
- "LoRA" vs. "Low-rank adaptation" scored only 0.40: the model doesn't know the acronym.
  Embeddings are weak on acronyms and exact terms, which is why hybrid search exists (Day 8).

## Step 5: Hallucination vs. RAG

**5a.** Same model, same question, with and without the paper's abstract in the prompt.

In [60]:
# Paste the abstract from https://arxiv.org/abs/2106.09685 between the triple quotes
lora_abstract = """An important paradigm of natural language processing consists of large-scale pre-training on general domain data and adaptation to particular tasks or domains. As we pre-train larger models, full fine-tuning, which retrains all model parameters, becomes less feasible. Using GPT-3 175B as an example -- deploying independent instances of fine-tuned models, each with 175B parameters, is prohibitively expensive. We propose Low-Rank Adaptation, or LoRA, which freezes the pre-trained model weights and injects trainable rank decomposition matrices into each layer of the Transformer architecture, greatly reducing the number of trainable parameters for downstream tasks. Compared to GPT-3 175B fine-tuned with Adam, LoRA can reduce the number of trainable parameters by 10,000 times and the GPU memory requirement by 3 times. LoRA performs on-par or better than fine-tuning in model quality on RoBERTa, DeBERTa, GPT-2, and GPT-3, despite having fewer trainable parameters, a higher training throughput, and, unlike adapters, no additional inference latency. We also provide an empirical investigation into rank-deficiency in language model adaptation, which sheds light on the efficacy of LoRA. We release a package that facilitates the integration of LoRA with PyTorch models and provide our implementations and model checkpoints for RoBERTa, DeBERTa, and GPT-2"""


def build_prompt(question: str, context: str) -> str:
    """Wrap a question with rules and context, so the model answers only from the context."""
    return f"""You answer questions using ONLY the context below.
If the context does not contain the answer, say: "I don't know based on the provided context."

Context:
{context}

Question: {question}
Answer:"""


question = "What does the acronym LoRA stand for, and who wrote the paper?"

print("WITHOUT context (closed book):")
print(ask(question))
print()
print("WITH context (open book):")
print(ask(build_prompt(question, lora_abstract)))

WITHOUT context (closed book):
seconds taken:     1.33
tokens generated:  60
tokens per second: 45.1
The acronym LoRA stands for "Local Reparameterization" or "Local Reparameterization with Additive Noise" (depending on the context).

The paper "Local Reparameterization for Efficient Neural Architecture Search" by Edwin Chu, et al. introduced the concept of LoRA.

WITH context (open book):
seconds taken:     0.72
tokens generated:  34
tokens per second: 47.3
Based on the provided context, the acronym LoRA stands for "Low-Rank Adaptation". However, the author of the paper is not mentioned in the context.


**5b.** RAG in miniature: embed the question, find the closest paragraph, and answer from it.

In [61]:
# A tiny "library": the real abstract plus three unrelated paragraphs
library = [
    lora_abstract,
    "Espresso is brewed by forcing hot water through finely ground coffee at high pressure. "
    "A standard shot takes about 25 to 30 seconds.",
    "The Federal Reserve raises interest rates to slow inflation. Higher rates make borrowing "
    "more expensive for both businesses and households.",
    "Transformers process all tokens in a sequence at once using self-attention, which lets "
    "each token weigh how relevant every other token is.",
]

question = "How can I fine-tune a large language model cheaply?"

# 1. Embed the question and every paragraph with the SAME model
question_embedding = model.encode(question, normalize_embeddings=True)
library_embeddings = model.encode(library, normalize_embeddings=True)

# 2. Score every paragraph against the question
scores = util.cos_sim(question_embedding, library_embeddings)[0]

for i in range(len(library)):
    print(f"{scores[i].item():.2f}  {library[i][:70]}...")

# 3. Pick the paragraph with the highest score
best_index = int(scores.argmax())
best_paragraph = library[best_index]
print()
print("Picked paragraph", best_index)

# 4. Answer using only that paragraph
print()
print(ask(build_prompt(question, best_paragraph)))

0.54  An important paradigm of natural language processing consists of large...
0.08  Espresso is brewed by forcing hot water through finely ground coffee a...
0.10  The Federal Reserve raises interest rates to slow inflation. Higher ra...
0.23  Transformers process all tokens in a sequence at once using self-atten...

Picked paragraph 0

seconds taken:     1.01
tokens generated:  46
tokens per second: 45.4
You can use Low-Rank Adaptation (LoRA), which freezes the pre-trained model weights and injects trainable rank decomposition matrices into each layer of the Transformer architecture, greatly reducing the number of trainable parameters for downstream tasks.


### Findings
- **Closed book:** asked what LoRA stands for, the model invented "Local Reparameterization" and a paper by "Edwin Chu et al.". Earlier, a slightly different question got a *different* invented answer ("Locally Optimized Weights…", "You et al., 2020"). Hallucinations aren't even consistent.
- **Open book:** with the abstract in the prompt, it answered **Low-Rank Adaptation** and said the authors weren't in the context instead of inventing them. The "say I don't know" rule in the prompt made that happen.
- **Retrieval worked on meaning:** "How can I fine-tune a large language model cheaply?" matched the LoRA abstract (0.54) even though it never says "cheaply". The transformers paragraph came second (0.23, related ML topic); espresso and the Fed scored about 0.1.
- **This is the whole RAG loop:** embed → find the closest text → put it in the prompt → answer only from it.

## Day 1 summary
| Step | Key lesson |
| --- | --- |
| 1. Tokens | Models read tokens, not words: about 0.75 words per token. Limits and chunk sizes are in tokens. |
| 2. Calling the LLM | Ollama is an HTTP API returning JSON. About 46 tokens/s on my Mac. The model confidently invented what LoRA means. |
| 3. Temperature | Controls randomness, not accuracy. T=0 gave the same answer 3/3 times; use it for RAG and evals. |
| 4. Embeddings | 384 numbers per sentence; paraphrases score about 0.6, unrelated text about 0.05. Weak on acronyms ("LoRA" vs. "Low-rank adaptation": 0.40). |
| 5. RAG | Giving the model the right text fixed the hallucination, and it admitted what the text didn't say. |